# पाठ १२ - एजेण्ट स्क्र्याचप्याडसँग कुरा इतिहासलाई घटाउने

यो नोटबुकले माइक्रोसफ्ट एजेण्ट फ्रेमवर्क प्रयोग गरी लामो कुराकानीमा सन्दर्भ कसरी व्यवस्थापन गर्ने देखाउँछ। जति कुराकानी बढ्छ, टोकन संख्या पनि बढ्छ — अन्ततः मोडलको सन्दर्भ विन्डो भन्दा बढी हुन्छ। हामी यसलाई **सन्दर्भ संक्षेपण ढाँचा** र **एजेण्ट स्क्र्याचप्याड** प्रयोग गरी दीर्घकालीन स्मृतिका लागि सम्बोधन गर्छौं।

## तपाईंले सिक्ने कुरा:  
१. **किन सन्दर्भ व्यवस्थापन महत्वपूर्ण छ**: टोकन सिमाना र सन्दर्भ विन्डो बुझ्नु  
२. **सन्दर्भ-जागरुक एजेन्टहरू**: आफ्नै कुराकानी सन्दर्भ व्यवस्थापन गर्ने एजेन्टहरू बनाउने  
३. **सन्दर्भ संक्षेपण ढाँचा**: कुराकानी इतिहास संक्षेप गर्न उपकरणहरूको प्रयोग  
४. **एजेण्ट स्क्र्याचप्याड**: सन्दर्भ घटाएपनि बाँच्ने दीर्घकालीन स्मृति  

## पूर्व आवश्यकताहरू:  
- वातावरण भेरिएबलहरू कन्फिगर गरिएको Azure OpenAI सेटअप  
- अघिल्ला पाठहरूबाट आधारभूत एजेन्ट अवधारणाको बुझाइ


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## किन सन्दर्भ व्यवस्थापन महत्त्वपूर्ण छ

हरेक LLM सँग सीमित **सन्दर्भ विन्डो** हुन्छ — एउटै अनुरोधमा प्रशोधन गर्न सकिने अधिकतम टोकनहरूको संख्या। बहु-परिवर्तनीय संवाद जाँदा:

- **टोकन गणना रैखिक रूपमा बाढ्छ** प्रत्येक प्रयोगकर्ता सन्देश र सहायकको जवाफसँगै।
- **प्रॉम्प्ट टोकनहरूले खर्चमा प्रभुत्व जमाउँछन्** किनकि सम्पूर्ण इतिहास हरेक पालोमा पुन: पठाइन्छ।
- अन्ततः संवाद **सन्दर्भ विन्डो पार गर्छ** र मोडेलले वा त काट्छ वा त्रुटि देखाउँछ।

### सन्दर्भ व्यवस्थापनका रणनीतिहरू

| रणनीति | कसरि काम गर्छ | ट्रेड-अफ |
|---|---|---|
| **काटफट** | पुराना सन्देशहरू हटाउनुहोस् | प्रारम्भिक सन्दर्भ हराउँछ |
| **सारांशकरण** | पुराना सन्देशहरूलाई संक्षेपमा रूपान्तरण गर्नुहोस् | केही विवरण हराउँछ, तर मुख्य बुँदाहरू राखिन्छन् |
| **स्क्र्याचप्याड / बाह्य स्मृति** | संवाद बाहिर मुख्य तथ्यहरू राख्नुहोस् | उपकरण कलहरूको आवश्यकता हुन्छ, तर कुनै पनि कटौतीबाट जोगिन्छ |

यस नोटबुकमा हामी **सारांशकरण** र **स्क्र्याचप्याड उपकरण** सँग संयोजन गर्छौं ताकि एजेन्टले संवाद इतिहास सङ्कुचित हुँदा समेत निरन्तरता राख्न सकोस्।


## सन्दर्भ-चेतित एजेन्ट सिर्जना गर्दै


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## लामो संवादलाई नक्कल गर्ने

सन्दर्भ कसरी संचय हुन्छ भनी हेर्नको लागि बहु-चरणीय संवादमा हिँडौं। एजेन्टले मुख्य विवरणहरू (रूचिहरू, बजेट, यात्रा मिति) प्रत्येक चरणमा राख्नुपर्छ र निरन्तरता देखाउनुपर्छ।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ध्यान दिनुहोस् कसरी एजेन्टले अघिल्लो संवादहरूबाट प्रसङ्ग राख्छ — यसले जापान, सुशी, मन्दिरहरू, फोटोग्राफी, $३००० को बजेट, एक्लो यात्रा, र अप्रिलको समयावधिको बारेमा जान्दछ। सानो कुराकानीमा यो राम्रोसँग काम गर्छ, तर जब कुराकानी बढ्छ तब पूरा इतिहास पुनः पठाउन महँगो हुन्छ।

आउनूहोस्, प्रसङ्गको संचय हेर्न थप संवादहरूका साथ कुराकानी जारी राखौं:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## सन्दर्भ संक्षेपण ढाँचा

संवाद बढ्दै जाँदा, हामीले सञ्चित सन्दर्भलाई संक्षिप्त प्रारूपमा सङ्कुचन गर्न **संक्षेपण उपकरण** प्रयोग गर्न सक्छौं। एजेन्टले यो उपकरणलाई मुख्य प्राथमिकताहरू रेकर्ड गर्न कल गर्छ ताकि पुराना सन्देशहरू हटाइए पनि आवश्यक जानकारी सुरक्षित रहोस्।

यो ढाँचा अझ परिष्कृत इतिहास संक्षेपणका लागि आधारभूत ब्लक हो:
1. एजेन्टले संवादबाट मुख्य तथ्यहरू पहिचान गर्छ
2. यसले तिनलाई सुरक्षित गर्न संक्षेपण उपकरणलाई कल गर्छ
3. पुराना सन्देशहरू सुरक्षित रूपमा हटाउन सकिन्छ किनभने सारांशले महत्त्वपूर्ण कुरा समेट्छ

तल हामीले `summarize_preferences` उपकरण परिभाषित गरेका छौं जुन एजेन्टले सिकेका कुराहरूको एक संक्षिप्त सारांश रेकर्ड गर्न कल गर्न सक्छ।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरी लामो समयसम्म चल्ने एजेन्ट संवादहरूमा सन्दर्भ कसरी व्यवस्थापन गर्ने भन्ने सिक्नुभयो:

### मुख्य अवधारणाहरू
- **सन्दर्भ विन्डोहरू सीमित हुन्छन्** — संवाद इतिहासमा प्रत्येक टोकनको खर्च हुन्छ र सीमा भित्र गणना गरिन्छ।
- **सारांश उपकरणहरूले** एजेन्टलाई संचित सन्दर्भलाई संक्षिप्त सारांशमा परिणत गर्न दिन्छ, जसले टोकन प्रयोग घटाउँछ र आवश्यक जानकारी कायम राख्छ।
- **एजेन्ट स्क्र्याचप्याडहरूले** कुनै पनि संवाद कटौती पछि पनि टिक्ने बाह्य स्मृति प्रदान गर्दछ।

### तपाईंले के बनाउनु भयो
- एक **सन्दर्भ-सचेत एजेन्ट** जसले बहु-वार्तालापहरूमा निरन्तरता कायम राख्छ
- एक **सारांश उपकरण** (`summarize_preferences`) जसले प्रयोगकर्ताका मुख्य विवरणहरू संक्षिप्त स्वरूपमा रेकर्ड गर्छ
- एक **बहु-वार्तालाप** जसले सन्दर्भ कायम राख्ने र परिवर्तन सम्हाल्ने प्रदर्शन गर्छ

### वास्तविक प्रयोगहरू
- **ग्राहक सेवा बोटहरू**: लामो समर्थन सेसनहरूमा प्राथमिकताहरू सम्झिन
- **व्यक्तिगत सहायकहरू**: सन्दर्भ पुनः व्याख्या नगरी चलिरहेको परियोजनाहरू ट्रयाक गर्न
- **शैक्षिक ट्यूटरहरू**: धेरै अन्तरक्रियाहरूमा विद्यार्थीको प्रगति कायम राख्न

### आगामी कदमहरू
- फाइल-आधारित टिकाउपनसहित पूर्ण स्क्र्याचप्याड उपकरण कार्यान्वयन गर्नुहोस्
- सारांश पछि स्वचालित इतिहास कटौती थप्नुहोस्
- सेमेंटिक मेमोरी खोजका लागि भेक्टर डेटाबेससँग संयोजन गर्नुहोस्
- पूर्ण सन्दर्भसहित दिनहरू पछि संवाद फेरि सुरु गर्न सक्ने एजेन्टहरू बनाउनुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यो कागजात AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताको लागि प्रयासरत छौँ, तर कृपया जान्नुहोस् कि स्वचालित अनुवादमा त्रुटि वा गलतता हुन सक्छ। मूल भाषा मा रहेको दस्तावेजलाई अधिकारिक स्रोत मान्नुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यो अनुवाद प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा व्याख्याका लागि हामी जिम्मेवार छैनौँ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
